# 2a. ERA5: model evaluation (calculations)

Two questions, asked of ERA5 and the LESFMIP ensembles over the years they share (1979–2013):

1. **Is each model consistent with ERA5?** Could ERA5 be one more member of a model's historical ensemble? (Part 1, this notebook)
2. **Is the observed change detectable, and which forcings is it consistent with?** (Part 2, notebook 02c)

This notebook works out part 1 and saves it in `RESULTS_DIR` (`results/tas/sample` or `results/tas/full` under `SCRATCH_DIR`); notebook 02b opens the results of both parts and draws the figures.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from extant import datatree, era5_evaluation as ev, observed_change as oc, preprocessing as pp
from extant.config import MAIN_EXPERIMENTS, SATELLITE_YEAR_RANGE, VARIABLES
from extant.datatree import skip_empty, tree_to_dataset
from extant.logger import setup_logging
from extant.plots import charts, forced_response as fr, methods
from plotting_modules.constants import FORCING_COLORS, SEASONS

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

#(t): Harmless warnings, silenced the same way in every notebook. zarr's, that parts of the stores (the text
#(t): coordinates, the consolidated metadata) aren't in the Zarr 3 standard yet:
warnings.filterwarnings('ignore', message='.*Zarr (format 3|V3) specification')
#(t): numpy's, about slices with nothing but NaN (land, a model without a member) or nothing at all, too few members for
#(t): a standard deviation, and dividing by zero or by NaN (the log of a probability of 0, say):
warnings.filterwarnings('ignore', message='All-NaN slice encountered', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0 for slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='(divide by zero|invalid value) encountered', category=RuntimeWarning)
#(t): dask's, about large task graphs, which are large on purpose:
warnings.filterwarnings('ignore', message='.*Sending large graph of size', category=UserWarning)
#(t): Every figure's text at the notebook's sizes and colours (fr.NOTEBOOK), as in every notebook
plt.rcParams.update(fr.NOTEBOOK.rc)

In [ ]:
%matplotlib inline

## Settings

The experiments (`MAIN_EXPERIMENTS`) and the years ERA5 and LESFMIP share (`SATELLITE_YEAR_RANGE`) are in `extant/config.py`, and the order of the seasons (`SEASONS`) in `plotting_modules/constants.py`, the same for every notebook. Only this notebook's own choices are here.

In [ ]:
#(t): The variable: its name and its units once notebook 01 has converted it (VARIABLE.name, VARIABLE.units)
VARIABLE = VARIABLES['tas']

In [ ]:
#(t): The grid point of the tests at one point (the 2.5° cell nearest config.POINT, on the Antarctic plateau)
SAMPLE_LAT_LON = dict(lat=-82.5, lon=120.0)

#(t): The model of the one-model quick looks
EXAMPLE_MODEL = 'CanESM5'

In [ ]:
#(t): Where notebook 01's seasonal means are, and where this notebook's results go: JASMIN's scratch
SCRATCH_DIR = Path('/work/scratch-pw4/aborow/ExtAnt')
SCRATCH_DIR

## Open the Data

Run **one** of the next two sections. Both leave `lesfmip_season_tree` (seasonal means, /<model>/<experiment>, on member × year × season × lat × lon), `era5_seasonal_da` (ERA5 on year × season × lat × lon, on the same grid), all in °C, and `RESULTS_DIR`, where the results are saved.

### Sample data

The 3×3 block of grid cells around the point in `tests/data` (two models), put through notebook 01's pre-processing.

In [ ]:
#(t): The sample's files, in this repository: <model>_<experiment>_sample.nc, monthly tas in K on (member, time, lat, lon)
SAMPLE_DIR = Path('../tests/data')
sample_files = sorted(SAMPLE_DIR.glob('*_*_sample.nc'))
sample_files

In [ ]:
#(t): The model and experiment of each file, from its name
sample_names = [path.stem.removesuffix('_sample').rsplit('_', 1) for path in sample_files]

In [ ]:
#(t): Each file of the experiments analysed opened, as a DataTree /<model>/<experiment>
lesfmip_monthly_tree = xr.DataTree.from_dict({f'{model}/{experiment}': xr.open_dataset(path).load()
                                              for path, (model, experiment) in zip(sample_files, sample_names)
                                              if experiment in MAIN_EXPERIMENTS})

In [ ]:
#(t): Seasonal means in °C, through notebook 01's pre-processing (pp.seasonal_tree: to °C, cut at 2014-12, complete
#(t): seasons in whole years)
lesfmip_season_tree = pp.seasonal_tree(lesfmip_monthly_tree, VARIABLE.name).compute()
lesfmip_season_tree

In [ ]:
#(t): ERA5's sample file: monthly tas in K from 1979, on (time, lat, lon)
era5_sample_path = SAMPLE_DIR / 'era5_sample.nc'
era5_monthly_da = xr.open_dataarray(era5_sample_path).load()
era5_sample_path

In [ ]:
#(t): Its seasonal means in °C, on the models' grid (pp.seasonal_era5, as in notebook 01)
era5_seasonal_da = pp.seasonal_era5(era5_monthly_da, VARIABLE.name, like=lesfmip_season_tree)
era5_seasonal_da

In [ ]:
#(t): Where the sample's results are saved
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'sample'
RESULTS_DIR

### Full data

The seasonal means notebook 01 saved (section 3) in `SCRATCH_DIR`. Opening is lazy.

In [ ]:
#(t): The two stores notebook 01 saved
seasonal_path = SCRATCH_DIR / 'seasonal' / f'lesfmip_{VARIABLE.name}.zarr'
era5_path = SCRATCH_DIR / 'seasonal' / f'era5_{VARIABLE.name}.zarr'
seasonal_path, era5_path

In [ ]:
#(t): Every model and experiment notebook 01 saved
lesfmip_saved_tree = xr.open_datatree(seasonal_path, engine='zarr', chunks={})

In [ ]:
#(t): Only the experiments analysed (filter keeps the model nodes above them)
lesfmip_season_tree = lesfmip_saved_tree.filter(lambda node: node.name in MAIN_EXPERIMENTS)
lesfmip_season_tree

In [ ]:
#(t): ERA5's seasonal means
era5_seasonal_da = xr.open_zarr(era5_path)[VARIABLE.name].load()
era5_seasonal_da

In [ ]:
#(t): Where the full data's results are saved
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'full'
RESULTS_DIR

### What was opened

Whichever section ran. An empty square is a model and experiment that was never opened.

In [ ]:
#(t): Members per model (rows) and experiment (columns)
member_counts_df = pd.Series({(node.parent.name, node.name): node.sizes['member']
                              for node in lesfmip_season_tree.leaves}).unstack()
member_counts_df = member_counts_df.reindex(columns=MAIN_EXPERIMENTS)

In [ ]:
#(t): The members: each model (rows) and experiment (columns), grey where there are none
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=1, n_legends=0, n_colorbars=1,
                                                  panel_width=0.9 * member_counts_df.shape[1] + 0.5,
                                                  panel_height=0.4 * member_counts_df.shape[0] + 0.3, left_margin=1.8),
                 layout='constrained')
axes = fig.add_gridspec(1, 1).subplots(squeeze=False)
counts_image = methods.plot_member_counts(member_counts_df, ax=axes[0, 0])
fig.colorbar(counts_image, ax=axes, location='bottom', shrink=0.6, label='Members')
fig.suptitle('LESFMIP members per model and experiment', x=0.01, ha='left');

In [ ]:
#(t): Each experiment's members at the point (one model, DJF): every member (thin) and the ensemble mean (thick), one
#(t): row each, the number of members beside it
model_point_tree = lesfmip_season_tree[EXAMPLE_MODEL].sel(**SAMPLE_LAT_LON).load()
point_experiments = [e for e in MAIN_EXPERIMENTS if e in model_point_tree.children]
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(point_experiments), n_cols=1, n_legends=0,
                                                  panel_width=8.5, panel_height=1.0, left_margin=1.4),
                 layout='constrained')
axes = fig.add_gridspec(len(point_experiments), 1).subplots(sharex=True, sharey=True, squeeze=False)
for ax, experiment in zip(axes[:, 0], point_experiments):
    experiment_members_da = model_point_tree[experiment][VARIABLE.name].sel(season='DJF')
    charts.plot_lines(experiment_members_da, 'year', ax=ax, color=FORCING_COLORS[experiment], linewidth=0.4, alpha=0.35)
    charts.plot_lines(experiment_members_da.mean('member'), 'year', ax=ax, color=FORCING_COLORS[experiment],
                      linewidth=2)
    ax.set_ylabel(f"{experiment}\n({experiment_members_da.sizes['member']})", rotation=0, ha='right', va='center')
axes[-1, 0].set_xlabel('Year')
fig.suptitle(f'{EXAMPLE_MODEL}, DJF, every member (thin) and the ensemble mean (thick), {VARIABLE.units}', x=0.01,
             ha='left');

## Climatologies

Every season: the 1979–2013 average (the climatology) and the year-to-year spread of ERA5 and of each model's historical ensemble, how they differ, the area mean through time, and one model's experiments. The models' climatology minus ERA5's is the mean-state bias, which only the tests of raw values see; every other test removes it.

### ERA5

In [ ]:
#(t): ERA5's climatology: its mean over SATELLITE_YEAR_RANGE in each season, on (season, lat, lon)
era5_climatology_da = era5_seasonal_da.sel(year=SATELLITE_YEAR_RANGE, season=SEASONS).mean('year')
era5_climatology_da

In [ ]:
era5_climatology_da.plot(x='lon', y='lat', col='season', cmap='RdBu_r');

In [ ]:
#(t): ERA5's year-to-year spread: the standard deviation of its seasonal means over SATELLITE_YEAR_RANGE
era5_std_da = era5_seasonal_da.sel(year=SATELLITE_YEAR_RANGE, season=SEASONS).std('year')

In [ ]:
era5_std_da.plot(x='lon', y='lat', col='season', cmap='Reds');

In [ ]:
#(t): ERA5 south of 60°S
era5_south_da = era5_seasonal_da.sel(season=SEASONS, lat=slice(None, -60))

In [ ]:
#(t): its area mean in every year (cos-latitude weighted)
era5_area_weights_da = np.cos(np.deg2rad(era5_south_da.lat))
era5_area_mean_da = era5_south_da.weighted(era5_area_weights_da).mean(['lat', 'lon'])

### Historical ensembles

In [ ]:
#(t): Each model's historical ensemble
historical_tree = lesfmip_season_tree.match('*/historical')

In [ ]:
#(t): over SATELLITE_YEAR_RANGE, in the seasons' order
historical_eval_tree = historical_tree.sel(year=SATELLITE_YEAR_RANGE, season=SEASONS)

In [ ]:
#(t): Each model's climatology: the mean over the years and the members
historical_climatology_tree = historical_eval_tree.mean(['year', 'member'])

In [ ]:
%%time
#(t): In one DataArray, on (model, season, lat, lon)
historical_climatology_ds = datatree.tree_to_dataset(historical_climatology_tree, dims=['model', 'experiment']).load()
historical_climatology_da = historical_climatology_ds[VARIABLE.name].sel(experiment='historical', drop=True)
historical_climatology_da

In [ ]:
historical_climatology_da.plot(x='lon', y='lat', row='model', col='season', cmap='RdBu_r');

In [ ]:
#(t): Each model's climatology minus ERA5's
historical_bias_da = historical_climatology_da - era5_climatology_da

In [ ]:
historical_bias_da.plot(x='lon', y='lat', row='model', col='season', cmap='RdBu_r', center=0);

In [ ]:
#(t): ERA5's climatology and each model's bias together, for 02b's first look (polar maps)
first_look_ds = xr.Dataset({'era5_mean': era5_climatology_da, 'model_bias': historical_bias_da})
first_look_ds

In [ ]:
#(t): Saved for 02b
first_look_path = RESULTS_DIR / 'first_look.zarr'
first_look_ds.to_zarr(first_look_path, mode='w')
first_look_path

In [ ]:
#(t): Each member's year-to-year standard deviation
historical_member_std_tree = historical_eval_tree.std('year')

In [ ]:
#(t): averaged over the members
historical_std_tree = historical_member_std_tree.mean('member')

In [ ]:
%%time
#(t): In one DataArray, on (model, season, lat, lon)
historical_std_ds = datatree.tree_to_dataset(historical_std_tree, dims=['model', 'experiment']).load()
historical_std_da = historical_std_ds[VARIABLE.name].sel(experiment='historical', drop=True)

In [ ]:
#(t): Each model's spread over ERA5's: above 1, the model varies more from year to year than ERA5
historical_std_ratio_da = historical_std_da / era5_std_da

In [ ]:
historical_std_ratio_da.plot(x='lon', y='lat', row='model', col='season', cmap='RdBu_r', center=1);

In [ ]:
#(t): Each model's historical ensemble mean, every year
historical_ensemble_mean_tree = historical_tree.mean('member')

In [ ]:
#(t): In one DataArray, on (model, year, season, lat, lon)
historical_ensemble_mean_ds = datatree.tree_to_dataset(historical_ensemble_mean_tree, dims=['model', 'experiment'])
historical_ensemble_mean_da = historical_ensemble_mean_ds[VARIABLE.name].sel(experiment='historical', drop=True)

In [ ]:
#(t): south of 60°S, in the seasons' order
historical_south_da = historical_ensemble_mean_da.sel(season=SEASONS, lat=slice(None, -60))

In [ ]:
%%time
#(t): its area mean in every year (cos-latitude weighted)
historical_area_weights_da = np.cos(np.deg2rad(historical_south_da.lat))
historical_area_mean_da = historical_south_da.weighted(historical_area_weights_da).mean(['lat', 'lon']).load()

In [ ]:
#(t): The models' ensemble means (colours) and ERA5 (black), one panel per season
fg = historical_area_mean_da.plot(x='year', hue='model', col='season')
for ax, season in zip(fg.axs.flat, SEASONS):
    era5_area_mean_da.sel(season=season).plot(ax=ax, color='k')

### One model's experiments

In [ ]:
#(t): EXAMPLE_MODEL's every experiment, over SATELLITE_YEAR_RANGE, in the seasons' order
model_eval_tree = lesfmip_season_tree[EXAMPLE_MODEL].sel(year=SATELLITE_YEAR_RANGE, season=SEASONS)

In [ ]:
#(t): Each experiment's climatology: the mean over the years and the members
model_climatology_tree = model_eval_tree.mean(['year', 'member'])

In [ ]:
%%time
#(t): In one DataArray, on (experiment, season, lat, lon)
model_climatology_ds = datatree.tree_to_dataset(model_climatology_tree, dims=['experiment']).load()
model_climatology_da = model_climatology_ds[VARIABLE.name]

In [ ]:
#(t): Each experiment's climatology minus hist-nat's, over the same years
model_difference_da = model_climatology_da - model_climatology_da.sel(experiment='hist-nat')

In [ ]:
#(t): Quick look: each experiment's climatology minus hist-nat's, every season
model_difference_da.drop_sel(experiment='hist-nat').plot(x='lon', y='lat', row='experiment', col='season',
                                                         cmap='RdBu_r', center=0);

# Part 1: Is Each Model Consistent with ERA5?

Every test asks one question: *could ERA5 be one more member of the historical ensemble?* If a model simulates both the forced response (the **signal**) and the internal variability (the **noise**) correctly, ERA5 is statistically indistinguishable from any one member. So each test computes the same statistic for ERA5 and for every member, and asks where ERA5 falls among the members (Suarez-Gutierrez, Milinski & Maher, 2021, *Clim. Dyn.*, [doi:10.1007/s00382-021-05821-w](https://doi.org/10.1007/s00382-021-05821-w)). The members are the null distribution, so no shape is assumed, and each member is exactly as long as ERA5.

First at one grid point, then on the maps (section 7 and 8) and in regional means (section 9).

## Data at the Point

Each model's historical members and ERA5 over `SATELLITE_YEAR_RANGE`. The models stay in a DataTree, each with its own members; ERA5 is one DataArray.

In [ ]:
#(t): LESFMIP over the years shared with ERA5
lesfmip_obs_tree = lesfmip_season_tree.sel(year=SATELLITE_YEAR_RANGE)

In [ ]:
#(t): Each model's historical members at the point, on (member, year, season)
historical_point_tree = lesfmip_obs_tree.match('*/historical').sel(SAMPLE_LAT_LON).load()
historical_point_tree

In [ ]:
#(t): ERA5 at the point over the same years, on (year, season)
era5_point_da = era5_seasonal_da.sel(year=SATELLITE_YEAR_RANGE).sel(SAMPLE_LAT_LON)
era5_point_da

In [ ]:
#(t): Check: every model has the same years as ERA5, and nothing is missing (the tests compare the same samples)
{node.parent.name: (int(node.year.min()), int(node.year.max()), int(node['tas'].isnull().sum()))
 for node in historical_point_tree.leaves}, (int(era5_point_da.year.min()), int(era5_point_da.year.max()),
                                             int(era5_point_da.isnull().sum()))

In [ ]:
#(t): Quick look: EXAMPLE_MODEL's members (grey) and ERA5 (black), every season
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5), sharex=True)
for ax, season in zip(axes, SEASONS):
    historical_point_tree[EXAMPLE_MODEL]['historical']['tas'].sel(season=season).plot.line(ax=ax, x='year',
                                                                                           hue='member',
                                                                                    add_legend=False, color='0.7', lw=0.5)
    era5_point_da.sel(season=season).plot(ax=ax, color='k')
    ax.set_title(season)

In [ ]:
#(t): Each model's hist-nat members at the point too, as context for 02b's time series
histnat_point_tree = lesfmip_obs_tree.match('*/hist-nat').sel(SAMPLE_LAT_LON).load()
histnat_point_tree

In [ ]:
#(t): Saved for 02b: each model's historical members at the point
historical_point_path = RESULTS_DIR / 'historical_point.zarr'
historical_point_tree.to_zarr(historical_point_path, mode='w')
historical_point_path

In [ ]:
#(t): ... their hist-nat members
histnat_point_path = RESULTS_DIR / 'histnat_point.zarr'
histnat_point_tree.to_zarr(histnat_point_path, mode='w')
histnat_point_path

In [ ]:
#(t): ... and ERA5
era5_point_path = RESULTS_DIR / 'era5_point.zarr'
era5_point_da.to_dataset(name='tas').to_zarr(era5_point_path, mode='w')
era5_point_path

## Three Treatments

Each series (every member and ERA5, separately, in each season) is treated in the same way, so they all lose the same things with the same sampling noise:

| Treatment | Removed from each series | A test of it then sees |
|---|---|---|
| raw values | nothing | the climatology; a mean-state bias dominates |
| anomalies | its own mean over the years | the forced trend (signal) and the variability (noise) |
| detrended | its own least-squares line | the variability (noise) only |

In [ ]:
#(t): Anomalies: each member minus its own mean over the years
historical_anomaly_tree = historical_point_tree - historical_point_tree.mean('year')

In [ ]:
#(t): ... and ERA5 minus its own
era5_anomaly_da = era5_point_da - era5_point_da.mean('year')

In [ ]:
#(t): Detrended: each member minus its own least-squares line through the years (ev.detrend)
historical_detrended_tree = historical_point_tree.map_over_datasets(skip_empty(ev.detrend))

In [ ]:
#(t): ... and ERA5 minus its own
era5_detrended_da = ev.detrend(era5_point_da)

In [ ]:
#(t): Quick look: the three treatments of EXAMPLE_MODEL's first member (grey) and ERA5 (black) at the point, in DJF
first_member_da = historical_point_tree[EXAMPLE_MODEL]['historical']['tas'].isel(member=0)
first_anomaly_da = historical_anomaly_tree[EXAMPLE_MODEL]['historical']['tas'].isel(member=0)
first_detrended_da = historical_detrended_tree[EXAMPLE_MODEL]['historical']['tas'].isel(member=0)
fig, axes = plt.subplots(1, 3, figsize=(16, 3), sharex=True, layout='constrained')
for ax, member_series_da, era5_series_da, treatment in zip(
        axes, [first_member_da, first_anomaly_da, first_detrended_da],
        [era5_point_da, era5_anomaly_da, era5_detrended_da], ['raw values', 'anomalies', 'detrended']):
    member_series_da.sel(season='DJF').plot(ax=ax, color='0.6')
    era5_series_da.sel(season='DJF').plot(ax=ax, color='k')
    ax.set_title(treatment)

## Where ERA5 Falls among the Members

Every test below ends the same way: the statistic of every model's members and ERA5's go to `ev.locate` (one model at a time, by going through the leaves of the tree), which returns for each season ERA5's value, the members' values and their 5%, 50% and 95%, ERA5's **percentile** among them (ties count half), the two-sided **p-value** (twice the smaller tail, ERA5 counted as one more member: 2 × (members at or beyond ERA5 on that side + 1) / (N + 1)), the **verdict** (−1 ERA5 below the members, +1 above, 0 not flagged; flagged when p < 0.1, i.e. ERA5 outside the members' 5–95%) and whether the model has enough members to flag anything (**testable**: N ≥ 20).

## Moment Test

The first four moments of each series: the **mean** of the raw values, and the **standard deviation**, **skewness** and **excess kurtosis** of the detrended values, so the 1979–2013 warming neither widens the spread nor bends the shape. To test a moment on other values, start its first cell from another tree (`historical_point_tree`, `historical_anomaly_tree` or `historical_detrended_tree`, and the matching ERA5), and change its treatment where the four are put together.

### Mean (raw values)

In [ ]:
#(t): Each member's mean over the years, on (member, season)
historical_mean_tree = historical_point_tree.mean('year')

In [ ]:
#(t): ERA5's
era5_mean_da = era5_point_da.mean('year')

In [ ]:
#(t): Where ERA5's falls among each model's members
mean_test_tree = xr.DataTree.from_dict({
    node.path: ev.locate(node['tas'], era5_mean_da) for node in historical_mean_tree.leaves
})

### Standard deviation (detrended)

In [ ]:
#(t): The standard deviation of each member's detrended values, on (member, season)
historical_std_tree = historical_detrended_tree.std('year', ddof=1)

In [ ]:
#(t): ERA5's
era5_std_da = era5_detrended_da.std('year', ddof=1)

In [ ]:
#(t): Where ERA5's falls among each model's members
std_test_tree = xr.DataTree.from_dict({
    node.path: ev.locate(node['tas'], era5_std_da) for node in historical_std_tree.leaves
})

### Skewness (detrended)

In [ ]:
#(t): ev.skewness: the plain moment estimator; > 0 when the warm tail is longer
historical_skewness_tree = historical_detrended_tree.map_over_datasets(skip_empty(ev.skewness))

In [ ]:
#(t): ERA5's
era5_skewness_da = ev.skewness(era5_detrended_da)

In [ ]:
#(t): Where ERA5's falls among each model's members
skewness_test_tree = xr.DataTree.from_dict({
    node.path: ev.locate(node['tas'], era5_skewness_da) for node in historical_skewness_tree.leaves
})

### Excess kurtosis (detrended)

In [ ]:
#(t): ev.excess_kurtosis: 0 for a normal distribution, > 0 for heavier tails
historical_kurtosis_tree = historical_detrended_tree.map_over_datasets(skip_empty(ev.excess_kurtosis))

In [ ]:
#(t): ERA5's
era5_kurtosis_da = ev.excess_kurtosis(era5_detrended_da)

In [ ]:
#(t): Where ERA5's falls among each model's members
kurtosis_test_tree = xr.DataTree.from_dict({
    node.path: ev.locate(node['tas'], era5_kurtosis_da) for node in historical_kurtosis_tree.leaves
})

### All four

In [ ]:
#(t): Each model's four tests along a statistic dim
moment_test_tree = xr.DataTree.from_dict({
    node.path: xr.concat([node.to_dataset(), std_test_tree[node.path].to_dataset(),
                          skewness_test_tree[node.path].to_dataset(), kurtosis_test_tree[node.path].to_dataset()],
                         dim=xr.Variable('statistic', ['mean', 'std', 'skewness', 'kurtosis']))
    for node in mean_test_tree.leaves
})

In [ ]:
#(t): With the treatment each moment was computed on, for 02b's labels
moment_test_tree = xr.DataTree.from_dict({
    node.path: node.to_dataset().assign_coords(treatment=('statistic', ['raw', 'detrended', 'detrended', 'detrended']))
    for node in moment_test_tree.leaves
})
moment_test_tree

In [ ]:
#(t): Quick look: EXAMPLE_MODEL's members (histogram) and ERA5 (black) for each moment, in DJF
example_moments_ds = moment_test_tree[EXAMPLE_MODEL]['historical'].to_dataset().sel(season='DJF')
fig, axes = plt.subplots(1, 4, figsize=(16, 2.5), layout='constrained')
for ax, statistic in zip(axes, example_moments_ds['statistic'].values):
    example_moments_ds['members'].sel(statistic=statistic).plot.hist(ax=ax, bins=20, color='0.7')
    ax.axvline(float(example_moments_ds['era5'].sel(statistic=statistic)), color='k')
    ax.set_title(f"{statistic} ({example_moments_ds['treatment'].sel(statistic=statistic).item()})")

In [ ]:
#(t): Saved for 02b
moment_test_path = RESULTS_DIR / 'moment_test.zarr'
moment_test_tree.to_zarr(moment_test_path, mode='w')
moment_test_path

## Other Statistics

Not moments, but tested in exactly the same way:

| Statistic | Of | What it measures |
|---|---|---|
| trend | raw values | the least-squares trend per decade: the forced response (**signal**) |
| emergence | raw values | the S/N against itself: the change of a 25-year LOWESS curve from the start of the record to the end, over the std of the deviations from it |
| width | detrended | Q95 − Q05, whose forced change the response notebooks analyse (**noise**) |
| lower_tail, upper_tail | detrended | the cold tail Q50 − Q05 and the warm tail Q95 − Q50 |
| records_high, records_low | raw values | how many record highs and lows (1 + 1/2 + … + 1/35 ≈ 4.1 of each with no change) |
| lag1 | detrended | the lag-1 autocorrelation: the year-to-year memory |

### Trend

In [ ]:
#(t): The least-squares trend, per decade
historical_trend_tree = historical_point_tree.map_over_datasets(skip_empty(ev.trend))

In [ ]:
#(t): ERA5's
era5_trend_da = ev.trend(era5_point_da)

In [ ]:
#(t): Where ERA5's falls among each model's members
trend_test_tree = xr.DataTree.from_dict({
    node.path: ev.locate(node['tas'], era5_trend_da) for node in historical_trend_tree.leaves
})

### Emergence

In [ ]:
#(t): The smoothed change over the record, over its own noise
historical_emergence_tree = historical_point_tree.map_over_datasets(skip_empty(ev.self_signal_to_noise))

In [ ]:
#(t): ERA5's
era5_emergence_da = ev.self_signal_to_noise(era5_point_da)

In [ ]:
#(t): Where ERA5's falls among each model's members
emergence_test_tree = xr.DataTree.from_dict({
    node.path: ev.locate(node['tas'], era5_emergence_da) for node in historical_emergence_tree.leaves
})

### Width

In [ ]:
#(t): Q95 - Q05
historical_width_tree = historical_detrended_tree.map_over_datasets(skip_empty(ev.quantile_width))

In [ ]:
#(t): ERA5's
era5_width_da = ev.quantile_width(era5_detrended_da)

In [ ]:
#(t): Where ERA5's falls among each model's members
width_test_tree = xr.DataTree.from_dict({
    node.path: ev.locate(node['tas'], era5_width_da) for node in historical_width_tree.leaves
})

### Lower tail

In [ ]:
#(t): Q50 - Q05
historical_lower_tail_tree = historical_detrended_tree.map_over_datasets(skip_empty(ev.lower_tail_width))

In [ ]:
#(t): ERA5's
era5_lower_tail_da = ev.lower_tail_width(era5_detrended_da)

In [ ]:
#(t): Where ERA5's falls among each model's members
lower_tail_test_tree = xr.DataTree.from_dict({
    node.path: ev.locate(node['tas'], era5_lower_tail_da) for node in historical_lower_tail_tree.leaves
})

### Upper tail

In [ ]:
#(t): Q95 - Q50
historical_upper_tail_tree = historical_detrended_tree.map_over_datasets(skip_empty(ev.upper_tail_width))

In [ ]:
#(t): ERA5's
era5_upper_tail_da = ev.upper_tail_width(era5_detrended_da)

In [ ]:
#(t): Where ERA5's falls among each model's members
upper_tail_test_tree = xr.DataTree.from_dict({
    node.path: ev.locate(node['tas'], era5_upper_tail_da) for node in historical_upper_tail_tree.leaves
})

### Records high

In [ ]:
#(t): Years warmer than every year before them
historical_records_high_tree = historical_point_tree.map_over_datasets(skip_empty(ev.record_highs))

In [ ]:
#(t): ERA5's
era5_records_high_da = ev.record_highs(era5_point_da)

In [ ]:
#(t): Where ERA5's falls among each model's members
records_high_test_tree = xr.DataTree.from_dict({
    node.path: ev.locate(node['tas'], era5_records_high_da) for node in historical_records_high_tree.leaves
})

### Records low

In [ ]:
#(t): Years colder than every year before them
historical_records_low_tree = historical_point_tree.map_over_datasets(skip_empty(ev.record_lows))

In [ ]:
#(t): ERA5's
era5_records_low_da = ev.record_lows(era5_point_da)

In [ ]:
#(t): Where ERA5's falls among each model's members
records_low_test_tree = xr.DataTree.from_dict({
    node.path: ev.locate(node['tas'], era5_records_low_da) for node in historical_records_low_tree.leaves
})

### Lag1

In [ ]:
#(t): The correlation of each year with the next, for every member of each model (ev.lag1_autocorrelation uses xr.corr,
#(t): which takes a DataArray, so it goes through the leaves of the tree)
historical_lag1_tree = xr.DataTree.from_dict({
    node.path: ev.lag1_autocorrelation(node['tas']).to_dataset(name='tas') for node in historical_detrended_tree.leaves
})

In [ ]:
#(t): ERA5's
era5_lag1_da = ev.lag1_autocorrelation(era5_detrended_da)

In [ ]:
#(t): Where ERA5's falls among each model's members
lag1_test_tree = xr.DataTree.from_dict({
    node.path: ev.locate(node['tas'], era5_lag1_da) for node in historical_lag1_tree.leaves
})

### All of them together

In [ ]:
#(t): Each model's eight tests along a statistic dim
statistic_test_tree = xr.DataTree.from_dict({
    node.path: xr.concat([node.to_dataset(), emergence_test_tree[node.path].to_dataset(),
                          width_test_tree[node.path].to_dataset(), lower_tail_test_tree[node.path].to_dataset(),
                          upper_tail_test_tree[node.path].to_dataset(), records_high_test_tree[node.path].to_dataset(),
                          records_low_test_tree[node.path].to_dataset(), lag1_test_tree[node.path].to_dataset()],
                         dim=xr.Variable('statistic', ['trend', 'emergence', 'width', 'lower_tail', 'upper_tail',
                                                       'records_high', 'records_low', 'lag1']))
    for node in trend_test_tree.leaves
})

In [ ]:
#(t): With the treatment of each, for 02b's labels
statistic_test_tree = xr.DataTree.from_dict({
    node.path: node.to_dataset().assign_coords(treatment=('statistic', ['raw', 'raw', 'detrended', 'detrended', 'detrended',
                                                                        'raw', 'raw', 'detrended']))
    for node in statistic_test_tree.leaves
})
statistic_test_tree

In [ ]:
#(t): Quick look: EXAMPLE_MODEL's members (histogram) and ERA5 (black) for each statistic, in DJF
example_statistics_ds = statistic_test_tree[EXAMPLE_MODEL]['historical'].to_dataset().sel(season='DJF')
fig, axes = plt.subplots(2, 4, figsize=(16, 5), layout='constrained')
for ax, statistic in zip(axes.flat, example_statistics_ds['statistic'].values):
    example_statistics_ds['members'].sel(statistic=statistic).plot.hist(ax=ax, bins=20, color='0.7')
    ax.axvline(float(example_statistics_ds['era5'].sel(statistic=statistic)), color='k')
    ax.set_title(f"{statistic} ({example_statistics_ds['treatment'].sel(statistic=statistic).item()})")

In [ ]:
#(t): Saved for 02b
statistic_test_path = RESULTS_DIR / 'statistic_test.zarr'
statistic_test_tree.to_zarr(statistic_test_path, mode='w')
statistic_test_path

## Rank Histograms

The test of Suarez-Gutierrez et al. (2021). Step by step on one model and season first, then `ev.rank_histogram_test` for every model.

1. Each year, ERA5's **rank**: how many members are below it.
2. The ranks of every year counted into a **histogram**. If ERA5 behaves like a member, every rank is equally likely and the histogram is flat.
3. With ~35 samples even a perfect model's histogram is not flat, so each member in turn is ranked against the other N − 1 (**pseudo-observations**): their histograms show how far from flat a perfect model's can be.
4. Each histogram summarised: the **dispersion** (the variance of the rank relative to a flat histogram: > 1 a U shape, too little spread; < 1 a dome, too much), how often ERA5 is outside every member, and the trend of the rank.
5. ERA5's summary located among the pseudo-observations'.

ERA5 is ranked against each of the N sub-ensembles that leave one member out, and those histograms averaged (`ev.leave_one_out_counts`), so it faces exactly the pseudo-observations' test.

In [ ]:
#(t): EXAMPLE_MODEL's anomalies and ERA5's in DJF, the example season
season_members_da = historical_anomaly_tree[EXAMPLE_MODEL]['historical']['tas'].sel(season='DJF')
season_era5_da = era5_anomaly_da.sel(season='DJF')

In [ ]:
#(t): 1. ERA5's rank in each year (0 to N)
season_era5_rank_da = ev.ensemble_rank(season_members_da, season_era5_da)

In [ ]:
#(t): 3. Each member's rank among the other N - 1, each year
season_pseudo_rank_da = ev.pseudo_observation_ranks(season_members_da)

In [ ]:
#(t): 2. and 3. The ranks counted over the years, on the same N bins
n_season_members = season_members_da.sizes['member']
season_era5_counts_da = ev.leave_one_out_counts(season_era5_rank_da, n_season_members, 'year')
season_pseudo_counts_da = ev.rank_counts(season_pseudo_rank_da, n_season_members, 'year')

In [ ]:
#(t): 4. Each histogram summarised
season_era5_summary_ds = ev.rank_statistics(season_era5_counts_da)
season_pseudo_summary_ds = ev.rank_statistics(season_pseudo_counts_da)

In [ ]:
#(t): 5. Where ERA5's dispersion falls among the pseudo-observations'
season_dispersion_test_ds = ev.locate(season_pseudo_summary_ds['dispersion'], season_era5_summary_ds['dispersion'])

In [ ]:
#(t): Quick look: the steps on EXAMPLE_MODEL's anomalies in DJF. ERA5's rank each year (1); the ranks counted,
#(t): ERA5's (black) and each pseudo-observation's (grey) (2 and 3); the dispersions, ERA5's (black) among the
#(t): pseudo-observations' (4 and 5)
fig, axes = plt.subplots(1, 3, figsize=(16, 3), layout='constrained')
season_era5_rank_da.plot(ax=axes[0], marker='o', color='k')
axes[0].set_title("1. ERA5's rank")
season_pseudo_counts_da.plot.line(ax=axes[1], x='rank', hue='member', add_legend=False, color='0.85')
season_era5_counts_da.plot.step(ax=axes[1], color='k')
axes[1].set_title('2 and 3. The ranks counted')
season_pseudo_summary_ds['dispersion'].plot.hist(ax=axes[2], bins=20, color='0.7')
axes[2].axvline(float(season_era5_summary_ds['dispersion']), color='k')
axes[2].set_title('4 and 5. Dispersion');

In [ ]:
#(t): The same through ev.rank_histogram_test, which treats the raw values itself and tests every summary
anatomy_result_ds = ev.rank_histogram_test(historical_point_tree[EXAMPLE_MODEL]['historical']['tas'].sel(season='DJF'),
                                           era5_point_da.sel(season='DJF'), 'anomaly')
anatomy_result_ds

In [ ]:
#(t): Check: both give the same dispersion test
check_variables = ['era5', 'percentile', 'pvalue', 'verdict']
pd.DataFrame({'step by step': season_dispersion_test_ds[check_variables].to_pandas(),
              'ev.rank_histogram_test': anatomy_result_ds.sel(statistic='dispersion')[check_variables].to_pandas()})

In [ ]:
#(t): The anomalies, together for 02b's figure of the method
rank_anatomy_ds = xr.Dataset({'members': season_members_da, 'era5': season_era5_da})

In [ ]:
#(t): Saved for 02b
rank_anatomy_data_path = RESULTS_DIR / 'rank_anatomy_data.zarr'
rank_anatomy_ds.to_zarr(rank_anatomy_data_path, mode='w')
rank_anatomy_data_path

In [ ]:
#(t): ... and the test
rank_anatomy_test_path = RESULTS_DIR / 'rank_anatomy_test.zarr'
anatomy_result_ds.to_zarr(rank_anatomy_test_path, mode='w')
rank_anatomy_test_path

### Every model

The headline histograms pool the four seasons (four times the samples); the scorecard uses one per season.

In [ ]:
#(t): Raw values, the four seasons pooled: ev.rank_histogram_test on each model's members (it needs each model's own
#(t): number of members), against ERA5
raw_ranks_tree = xr.DataTree.from_dict({
    node.path: ev.rank_histogram_test(node['tas'], era5_point_da, 'raw', pool_dims=('year', 'season'))
    for node in historical_point_tree.leaves
})

In [ ]:
#(t): Anomalies, the four seasons pooled
anomaly_ranks_tree = xr.DataTree.from_dict({
    node.path: ev.rank_histogram_test(node['tas'], era5_point_da, 'anomaly', pool_dims=('year', 'season'))
    for node in historical_point_tree.leaves
})
anomaly_ranks_tree

In [ ]:
#(t): Detrended anomalies, the four seasons pooled
detrended_ranks_tree = xr.DataTree.from_dict({
    node.path: ev.rank_histogram_test(node['tas'], era5_point_da, 'detrended', pool_dims=('year', 'season'))
    for node in historical_point_tree.leaves
})

In [ ]:
#(t): Quick look: EXAMPLE_MODEL's histogram of anomalies (bars: ERA5; grey: each pseudo-observation's)
anomaly_ranks_ds = anomaly_ranks_tree[EXAMPLE_MODEL]['historical'].to_dataset()
anomaly_ranks_ds['pseudo_frequency'].plot.line(x='rank', hue='member', add_legend=False, color='0.85')
anomaly_ranks_ds['obs_frequency'].plot.step(color='k');

In [ ]:
#(t): One test per season, for the scorecard (without the histograms): anomalies
anomaly_ranks_seasonal_tree = xr.DataTree.from_dict({
    node.path: ev.rank_histogram_test(node['tas'], era5_point_da, 'anomaly', pool_dims=('year',), keep_histograms=False)
    for node in historical_point_tree.leaves
})

In [ ]:
#(t): ... and detrended anomalies
detrended_ranks_seasonal_tree = xr.DataTree.from_dict({
    node.path: ev.rank_histogram_test(node['tas'], era5_point_da, 'detrended', pool_dims=('year',),
                                      keep_histograms=False)
    for node in historical_point_tree.leaves
})

In [ ]:
#(t): Saved for 02b
raw_ranks_path = RESULTS_DIR / 'raw_ranks.zarr'
raw_ranks_tree.to_zarr(raw_ranks_path, mode='w')
raw_ranks_path

In [ ]:
#(t): Saved for 02b
anomaly_ranks_path = RESULTS_DIR / 'anomaly_ranks.zarr'
anomaly_ranks_tree.to_zarr(anomaly_ranks_path, mode='w')
anomaly_ranks_path

In [ ]:
#(t): Saved for 02b
detrended_ranks_path = RESULTS_DIR / 'detrended_ranks.zarr'
detrended_ranks_tree.to_zarr(detrended_ranks_path, mode='w')
detrended_ranks_path

## Every Test in One Table

Each test's percentile, verdict and whether it could flag anything, without the members, so every model fits in one Dataset on (model, statistic, season): for 02b's scorecard and signal-vs-noise diagram. Both rank tests have a dispersion, so those two are named for their treatment.

In [ ]:
#(t): The moments' verdicts only (no members), every model, on (model, experiment, statistic, season)
moment_verdicts_tree = xr.DataTree.from_dict({node.path: node.to_dataset()[['percentile', 'verdict', 'testable']]
                                              for node in moment_test_tree.leaves})
moment_verdicts_ds = tree_to_dataset(moment_verdicts_tree, dims=['model', 'experiment'])

In [ ]:
#(t): ... and the other statistics'
statistic_verdicts_tree = xr.DataTree.from_dict({node.path: node.to_dataset()[['percentile', 'verdict', 'testable']]
                                                 for node in statistic_test_tree.leaves})
statistic_verdicts_ds = tree_to_dataset(statistic_verdicts_tree, dims=['model', 'experiment'])

In [ ]:
#(t): The rank trend and dispersion of the anomalies, verdicts only
anomaly_rank_verdicts_tree = xr.DataTree.from_dict({node.path: node.to_dataset()[['percentile', 'verdict', 'testable']]
                                                    for node in anomaly_ranks_seasonal_tree.leaves})
anomaly_rank_verdicts_ds = tree_to_dataset(anomaly_rank_verdicts_tree, dims=['model', 'experiment'])
anomaly_rank_verdicts_ds = anomaly_rank_verdicts_ds.sel(statistic=['rank_trend', 'dispersion']).drop_vars(['label', 'units'])
anomaly_rank_verdicts_ds = anomaly_rank_verdicts_ds.assign_coords(statistic=['rank_trend', 'dispersion (anomaly)'])

In [ ]:
#(t): ... and the dispersion of the detrended anomalies
detrended_rank_verdicts_tree = xr.DataTree.from_dict({node.path: node.to_dataset()[['percentile', 'verdict', 'testable']]
                                                      for node in detrended_ranks_seasonal_tree.leaves})
detrended_rank_verdicts_ds = tree_to_dataset(detrended_rank_verdicts_tree, dims=['model', 'experiment'])
detrended_rank_verdicts_ds = detrended_rank_verdicts_ds.sel(statistic=['dispersion']).drop_vars(['label', 'units'])
detrended_rank_verdicts_ds = detrended_rank_verdicts_ds.assign_coords(statistic=['dispersion (detrended)'])

In [ ]:
#(t): Everything along one statistic dim
eval_table_by_experiment_ds = xr.concat([moment_verdicts_ds, statistic_verdicts_ds, anomaly_rank_verdicts_ds,
                                         detrended_rank_verdicts_ds], dim='statistic')
eval_table_ds = eval_table_by_experiment_ds.sel(experiment='historical', drop=True)
#(c): tree_to_dataset leaves the model names as Python objects, which xarray will not put on an axis
eval_table_ds = eval_table_ds.assign_coords(model=eval_table_ds['model'].astype(str))
eval_table_ds

In [ ]:
#(t): Quick look: ERA5's percentile among each model's members (50: in the middle; 0 or 100: beyond every member)
eval_table_ds['percentile'].sel(season=SEASONS).plot(x='season', y='model', col='statistic', col_wrap=5, vmin=0,
                                                     vmax=100, cmap='RdBu_r');

In [ ]:
#(t): Saved for 02b
eval_table_path = RESULTS_DIR / 'eval_table.zarr'
eval_table_ds.to_zarr(eval_table_path, mode='w')
eval_table_path

## Toy Ensembles with a Known Answer

Before trusting the tests on real data: synthetic ensembles where we *know* what is wrong (`ev.simulate_ensemble`: 50 members and a pseudo-ERA5, 1979–2013, four seasons, each series trend × decades + noise × N(0, 1)). Pseudo-ERA5 always has a trend of 0.3 per decade and noise 1; each scenario of `ev.SCENARIOS` changes one thing about the model. 02b draws them.

In [ ]:
#(t): A toy ensemble and pseudo-ERA5 for each scenario, side by side on a scenario dim (all have 50 members)
demo_datasets = []
for settings in ev.SCENARIOS.values():
    demo_members_da, demo_era5_da = ev.simulate_ensemble(seed=3, **settings)
    demo_datasets.append(xr.Dataset({'members': demo_members_da, 'era5': demo_era5_da}))
demo_ds = xr.concat(demo_datasets, dim=xr.Variable('scenario', list(ev.SCENARIOS)))

In [ ]:
#(t): Quick look: each scenario's members (grey) and pseudo-ERA5 (black) in DJF
fg = demo_ds['members'].sel(season='DJF').plot.line(x='year', hue='member', col='scenario', add_legend=False,
                                                    color='0.7', lw=0.4)
for ax, scenario in zip(fg.axs.flat, demo_ds.scenario.values):
    demo_ds['era5'].sel(season='DJF', scenario=scenario).plot(ax=ax, color='k')

In [ ]:
#(t): The detrended values
demo_detrended_ds = demo_ds.map(ev.detrend)

In [ ]:
#(t): Four statistics of each, and where pseudo-ERA5's falls: the mean and trend of the raw values, the std and width
#(t): of the detrended values
demo_mean_ds = demo_ds.mean('year')
demo_trend_ds = demo_ds.map(ev.trend)
demo_std_ds = demo_detrended_ds.std('year', ddof=1)
demo_width_ds = demo_detrended_ds.map(ev.quantile_width)

In [ ]:
demo_tests_ds = xr.concat([ev.locate(demo_mean_ds['members'], demo_mean_ds['era5']),
                           ev.locate(demo_trend_ds['members'], demo_trend_ds['era5']),
                           ev.locate(demo_std_ds['members'], demo_std_ds['era5']),
                           ev.locate(demo_width_ds['members'], demo_width_ds['era5'])],
                          dim=xr.Variable('statistic', ['mean', 'trend', 'std', 'width']))
demo_tests_ds['percentile'].sel(season='DJF').plot(x='scenario', y='statistic', vmin=0, vmax=100, cmap='RdBu_r');

In [ ]:
#(t): The rank-histogram tests, the four seasons pooled: anomalies, and detrended anomalies
demo_anomaly_ranks_ds = ev.rank_histogram_test(demo_ds['members'], demo_ds['era5'], 'anomaly',
                                               pool_dims=('year', 'season'))
demo_detrended_ranks_ds = ev.rank_histogram_test(demo_ds['members'], demo_ds['era5'], 'detrended',
                                                 pool_dims=('year', 'season'))

In [ ]:
#(t): The plume of the raw values: the members' quantiles each year, and where pseudo-ERA5 is
demo_plume_ds = ev.ensemble_plume(demo_ds['members'], demo_ds['era5'], 'raw')

In [ ]:
#(t): Saved for 02b, without the members
demo_tests_path = RESULTS_DIR / 'demo_tests.zarr'
demo_tests_ds.drop_dims('member').to_zarr(demo_tests_path, mode='w')
demo_tests_path

In [ ]:
#(t): Saved for 02b
demo_anomaly_ranks_path = RESULTS_DIR / 'demo_anomaly_ranks.zarr'
demo_anomaly_ranks_ds.to_zarr(demo_anomaly_ranks_path, mode='w')
demo_anomaly_ranks_path

In [ ]:
#(t): Saved for 02b
demo_detrended_ranks_path = RESULTS_DIR / 'demo_detrended_ranks.zarr'
demo_detrended_ranks_ds.to_zarr(demo_detrended_ranks_path, mode='w')
demo_detrended_ranks_path

In [ ]:
#(t): Saved for 02b
demo_plume_path = RESULTS_DIR / 'demo_plume.zarr'
demo_plume_ds.to_zarr(demo_plume_path, mode='w')
demo_plume_path

## 7. Moment Test Maps

The moment test of the point, at every grid point of every model. It takes minutes, so it saves its result; the *reopen* cell is for next time. Then the **field test** (`ev.field_test`): a perfect model is flagged at about 10% of grid points by chance, and neighbouring points are correlated, so the question is whether ERA5 is flagged over *more* of the map than a perfect model would be. Each member in turn plays ERA5 against the other N − 1, and the areas they are flagged over show what a perfect model gives (Livezey & Chen, 1983, with the members as the null).

### Calculate (slow; saves the result)

In [ ]:
#(t): The moment test at every grid point of one model, by the same steps as at the point, then the field test
#(c): One model's 1979-2013 field (~100 MB) is loaded at a time. The members are dropped once tested, so models of
#(c): any ensemble size stack on one grid
def map_moment_test(ds):
    """The moment test at every grid point of one model's historical members, and the field test over the map.

    Args:
        ds (xr.Dataset): One model's node: tas on (member, year, season, lat, lon), over SATELLITE_YEAR_RANGE.

    Returns:
        xr.Dataset: ev.locate's results (without the members) on (statistic, season, lat, lon), and ev.field_test's
        on (statistic, season).
    """
    members_da = ds['tas'].load()
    era5_da = era5_seasonal_da.sel(year=SATELLITE_YEAR_RANGE)
    #(c): The standard deviation, skewness and kurtosis are of the detrended values, as at the point
    members_detrended_da = ev.detrend(members_da)
    era5_detrended_da = ev.detrend(era5_da)

    mean_test_ds = ev.locate(members_da.mean('year'), era5_da.mean('year'))
    std_test_ds = ev.locate(members_detrended_da.std('year', ddof=1), era5_detrended_da.std('year', ddof=1))
    skewness_test_ds = ev.locate(ev.skewness(members_detrended_da), ev.skewness(era5_detrended_da))
    kurtosis_test_ds = ev.locate(ev.excess_kurtosis(members_detrended_da), ev.excess_kurtosis(era5_detrended_da))
    moments_ds = xr.concat([mean_test_ds, std_test_ds, skewness_test_ds, kurtosis_test_ds],
                           dim=xr.Variable('statistic', ['mean', 'std', 'skewness', 'kurtosis']))

    field_test_ds = ev.field_test(moments_ds)
    return xr.merge([moments_ds, field_test_ds]).drop_dims('member')

In [ ]:
%%time
#(t): Example: EXAMPLE_MODEL's maps
example_map_test_ds = map_moment_test(lesfmip_obs_tree[EXAMPLE_MODEL]['historical'].to_dataset())
example_map_test_ds

In [ ]:
#(t): Check: at the point, the maps give the point's moment test (largest difference of ERA5's percentile: 0)
float(abs(example_map_test_ds['percentile'].sel(SAMPLE_LAT_LON)
          - moment_test_tree[EXAMPLE_MODEL]['historical']['percentile']).max())

In [ ]:
%%time
#(t): For every model, on (model, statistic, season, lat, lon)
eval_map_by_experiment_ds = datatree.reduce_to_dataset(lesfmip_obs_tree.match('*/historical'), map_moment_test,
                                                       coords='minimal', compat='override')
eval_map_table_ds = eval_map_by_experiment_ds.sel(experiment='historical', drop=True)

In [ ]:
#(t): Saved for 02b
eval_map_table_path = RESULTS_DIR / 'eval_map_table.zarr'
eval_map_table_ds.to_zarr(eval_map_table_path, mode='w')
eval_map_table_path

### Or reopen the saved result

In [ ]:
eval_map_table_ds = xr.open_zarr(RESULTS_DIR / 'eval_map_table.zarr').load()

### What was made

In [ ]:
eval_map_table_ds

In [ ]:
#(t): Check: at the point the maps give the point's moment test (True if every percentile agrees)
moment_percentile_da = tree_to_dataset(moment_verdicts_tree, dims=['model', 'experiment'])['percentile']
moment_percentile_da = moment_percentile_da.sel(experiment='historical', drop=True)
map_point_percentile_da = eval_map_table_ds['percentile'].sel(SAMPLE_LAT_LON).sel(model=moment_percentile_da.model)
bool(np.allclose(map_point_percentile_da.transpose(*moment_percentile_da.dims), moment_percentile_da, equal_nan=True))

In [ ]:
#(t): Quick look: ERA5's percentile among EXAMPLE_MODEL's members for the standard deviation, every season
eval_map_table_ds['percentile'].sel(statistic='std', model=EXAMPLE_MODEL).plot(x='lon', y='lat', col='season',
                                                                                vmin=0, vmax=100, cmap='RdBu_r');

## 8. Spatial Evaluation (Suarez-Gutierrez et al., 2021)

The paper's grid-point evaluation (their sections 2.2.2 and 3.2–3.4), at every grid point and season, by `ev.spatial_evaluation`: each year, is ERA5 (in anomalies) below every member, above every member, or in their central 75%? A perfect model puts ERA5 beyond each edge in 100/(N + 1)% of years and in the middle about 75% of the time; the paper flags ≥ 10% below, ≥ 10% above or > 80% in the middle. When the problem years fall (1979–1996 or 1997–2013) then says whether the forced response or the variability is wrong. 02b draws the maps.

### Calculate (slow; saves the result)

In [ ]:
#(t): The paper's thresholds (10% / 10% / 80%), and the two halves of the record
SPATIAL_THRESHOLDS = 'fixed'
SPATIAL_PERIODS = {'early': slice(1979, 1996), 'late': slice(1997, None)}

In [ ]:
#(t): One model: the whole record, each half, and whether a problem is spread evenly over the two (the variability)
#(t): or not (the forced response)
def spatial_evaluation(ds):
    """The paper's grid-point evaluation (ev.spatial_evaluation) of one model's historical members against ERA5.

    Args:
        ds (xr.Dataset): One model's node: tas on (member, year, season, lat, lon), over SATELLITE_YEAR_RANGE.

    Returns:
        xr.Dataset: ev.spatial_evaluation's result over the whole record on (season, lat, lon), with
        period_diagnosis (ev.period_comparison of the two halves, SPATIAL_PERIODS).
    """
    members_da = ds['tas'].load()
    era5_da = era5_seasonal_da.sel(year=SATELLITE_YEAR_RANGE)
    whole_ds = ev.spatial_evaluation(members_da, era5_da, thresholds=SPATIAL_THRESHOLDS)
    early_ds = ev.spatial_evaluation(members_da, era5_da, period=SPATIAL_PERIODS['early'], thresholds=SPATIAL_THRESHOLDS)
    late_ds = ev.spatial_evaluation(members_da, era5_da, period=SPATIAL_PERIODS['late'], thresholds=SPATIAL_THRESHOLDS)
    return whole_ds.assign(period_diagnosis=ev.period_comparison(whole_ds, early_ds, late_ds))

In [ ]:
%%time
#(t): Example: EXAMPLE_MODEL
example_spatial_ds = spatial_evaluation(lesfmip_obs_tree[EXAMPLE_MODEL]['historical'].to_dataset())
example_spatial_ds

In [ ]:
#(t): Check at the point, by hand: the % of years ERA5's anomaly is below every member's (each series minus its own
#(t): mean over the years), against the example's
point_member_anomaly_da = historical_anomaly_tree[EXAMPLE_MODEL]['historical']['tas']
point_below_da = 100 * (era5_anomaly_da < point_member_anomaly_da.min('member')).mean('year')
float(abs(point_below_da - example_spatial_ds['below'].sel(SAMPLE_LAT_LON)).max())

In [ ]:
%%time
spatial_by_experiment_ds = datatree.reduce_to_dataset(lesfmip_obs_tree.match('*/historical'), spatial_evaluation,
                                                      coords='minimal', compat='override')
spatial_table_ds = spatial_by_experiment_ds.sel(experiment='historical', drop=True)

In [ ]:
#(t): Saved for 02b
spatial_table_path = RESULTS_DIR / 'spatial_table.zarr'
spatial_table_ds.to_zarr(spatial_table_path, mode='w')
spatial_table_path

### Or reopen the saved result

In [ ]:
spatial_table_ds = xr.open_zarr(RESULTS_DIR / 'spatial_table.zarr').load()

### What was made

In [ ]:
spatial_table_ds

In [ ]:
#(t): Quick look: the % of years ERA5 is below every member of EXAMPLE_MODEL, every season
spatial_table_ds['below'].sel(model=EXAMPLE_MODEL).plot(x='lon', y='lat', col='season', vmin=0, vmax=30);

## 9. Regional Evaluation

A regional mean averages away much of a grid point's noise, which is why the paper's verification is regional:

- ERA5's regional means: `era5_regional`;
- the rank-histogram test of each region's mean, every model and experiment, anomalies and detrended: `regional_ranks`;
- ERA5's 1979–2013 trend among the members of every experiment: `regional_consistency`;
- ERA5's trend map, and each model's ensemble-mean trend in every experiment: `trend_maps`.

In [ ]:
#(t): The regions (observed_change.REGIONS), as in part 2 (notebook 02c)
oc.REGIONS

In [ ]:
#(t): ERA5's regional means, on (region, year, season)
era5_regional_da = oc.regional_means(era5_seasonal_da).rename(VARIABLE.name)
era5_regional_da

In [ ]:
#(t): Saved for notebook 07
era5_regional_path = RESULTS_DIR / 'era5_regional.zarr'
era5_regional_da.to_dataset(name=VARIABLE.name).to_zarr(era5_regional_path, mode='w')
era5_regional_path

In [ ]:
%%time
#(t): Every member's regional means over the ERA5 years, every model and experiment
regional_obs_tree = oc.regional_means(lesfmip_obs_tree).compute()

In [ ]:
#(t): Quick look: ERA5 (black) among EXAMPLE_MODEL's historical members (grey) in the first region, every season
regional_ensemble_da = regional_obs_tree[EXAMPLE_MODEL]['historical'][VARIABLE.name].isel(region=0)
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5), sharex=True)
for ax, season in zip(axes, ev.SEASONS):
    ax.plot(regional_ensemble_da.year, regional_ensemble_da.sel(season=season).transpose('year', 'member'), color='0.7', lw=0.5)
    ax.plot(era5_regional_da.year, era5_regional_da.isel(region=0).sel(season=season), color='k')
    ax.set_title(f"{str(regional_ensemble_da.region.values)}, {season}")

In [ ]:
%%time
#(t): Every model and experiment, regional means, anomalies: ev.rank_histogram_summary of each one's members against
#(t): ERA5's regional means over SATELLITE_YEAR_RANGE
regional_ranks_anomaly_tree = xr.DataTree.from_dict({
    node.path: ev.rank_histogram_summary(node['tas'], era5_regional_da.sel(year=SATELLITE_YEAR_RANGE), 'anomaly')
    for node in regional_obs_tree.leaves
})
regional_ranks_anomaly_ds = tree_to_dataset(regional_ranks_anomaly_tree, dims=['model', 'experiment'],
                                            coords='minimal', compat='override')

In [ ]:
%%time
#(t): ... and detrended anomalies
regional_ranks_detrended_tree = xr.DataTree.from_dict({
    node.path: ev.rank_histogram_summary(node['tas'], era5_regional_da.sel(year=SATELLITE_YEAR_RANGE), 'detrended')
    for node in regional_obs_tree.leaves
})
regional_ranks_detrended_ds = tree_to_dataset(regional_ranks_detrended_tree, dims=['model', 'experiment'],
                                              coords='minimal', compat='override')

In [ ]:
#(t): The two along a treatment dim
regional_ranks_ds = xr.concat([regional_ranks_anomaly_ds.drop_vars('treatment'),
                               regional_ranks_detrended_ds.drop_vars('treatment')],
                              dim=xr.Variable('treatment', ['anomaly', 'detrended']))
regional_ranks_ds

In [ ]:
#(t): Saved for notebook 07
regional_ranks_path = RESULTS_DIR / 'regional_ranks.zarr'
regional_ranks_ds.to_zarr(regional_ranks_path, mode='w')
regional_ranks_path

In [ ]:
#(t): Quick look: EXAMPLE_MODEL's historical histograms (anomalies) in the first region, ERA5 against the perfect-model
#(t): range
regional_ranks_ds[['obs_frequency', 'pseudo_lower', 'pseudo_upper']].sel(
    model=EXAMPLE_MODEL, experiment='historical', treatment='anomaly').isel(region=0).to_dataarray('line').plot.line(
    x='rank', hue='line', col='season');

In [ ]:
%%time
#(t): Every member's regional trend (per decade), every model and experiment
regional_members_trend_tree = regional_obs_tree.map_over_datasets(skip_empty(ev.trend))

In [ ]:
#(t): ERA5's regional trends over SATELLITE_YEAR_RANGE
era5_regional_trend_da = ev.trend(era5_regional_da.sel(year=SATELLITE_YEAR_RANGE))

In [ ]:
#(t): Where ERA5's falls among each model's and experiment's members (the members dropped)
regional_trend_tree = xr.DataTree.from_dict({
    node.path: ev.locate(node['tas'], era5_regional_trend_da).drop_dims('member')
    for node in regional_members_trend_tree.leaves
})

In [ ]:
#(t): On (statistic, model, experiment, region, season), the statistic named, without the labels, as notebook 07 opens it
regional_trend_ds = tree_to_dataset(regional_trend_tree, dims=['model', 'experiment'])
regional_consistency_ds = regional_trend_ds.expand_dims(statistic=['trend'])
regional_consistency_ds = regional_consistency_ds.drop_vars(['label', 'units', 'treatment'], errors='ignore')
regional_consistency_ds

In [ ]:
#(t): Saved for notebook 07
regional_consistency_path = RESULTS_DIR / 'regional_consistency.zarr'
regional_consistency_ds.to_zarr(regional_consistency_path, mode='w')
regional_consistency_path

In [ ]:
#(t): Quick look: ERA5's trend percentile among each experiment's members, EXAMPLE_MODEL, every region and season
consistency_percentile_da = regional_consistency_ds['percentile'].sel(model=EXAMPLE_MODEL, statistic='trend',
                                                                      season=SEASONS)
consistency_percentile_da.assign_coords(experiment=consistency_percentile_da['experiment'].astype(str)).plot(
    x='season', y='experiment', col='region', vmin=0, vmax=100, cmap='RdBu_r');

In [ ]:
#(t): The years ERA5 and the models share (LESFMIP ends with DJF 2013)
trend_years_arr = np.intersect1d(era5_seasonal_da['year'], lesfmip_obs_tree[EXAMPLE_MODEL]['historical']['year'])
trend_years_arr[[0, -1]]

In [ ]:
#(t): ERA5's trend over them at every grid point (per decade)
era5_trend_da = ev.trend(era5_seasonal_da.sel(year=trend_years_arr))

In [ ]:
%%time
#(t): Every model's ensemble-mean trend over the same years, every experiment
model_trend_ds = datatree.reduce_to_dataset(lesfmip_obs_tree.sel(year=trend_years_arr),
                                            lambda ds: ds.mean('member').map(ev.trend)).compute()

In [ ]:
#(t): The trend maps in one Dataset, laid out as notebook 07 opens them
trend_maps_ds = xr.Dataset({'era5_trend': era5_trend_da, 'model_trend': model_trend_ds[VARIABLE.name]})
trend_maps_ds

In [ ]:
#(t): Saved for notebook 07
trend_maps_path = RESULTS_DIR / 'trend_maps.zarr'
trend_maps_ds.to_zarr(trend_maps_path, mode='w')
trend_maps_path

In [ ]:
#(t): Quick look: ERA5's trend, every season
trend_maps_ds['era5_trend'].sel(season=SEASONS).plot(x='lon', y='lat', col='season', cmap='RdBu_r', center=0);